# FunctionGemma Dynamic LoRA with LiteRT-LM

<table class="tfo-notebook-buttons">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_2_LoRA_on_LiteRT_LM.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_2_LoRA_on_LiteRT_LM.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
</table>

[LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) is Google’s high-performance edge inference engine that runs LLMs across mobile, web, and desktop with full CPU/GPU/NPU acceleration and production-ready SDKs.

This notebook shows **dynamic LoRA deployment and runtime adapter swapping** with LiteRT-LM (`litert-lm`) and [FunctionGemma 270M](https://huggingface.co/google/functiongemma-270m-it). Unlike static LoRA fusion (which permanently merges adapter weights into a full copy of the base model), **dynamic LoRA** ships the base `.litertlm` model once with open low-rank input slots and attaches lightweight `.tflite` LoRA adapters at runtime per conversation.

It uses the smart-home tool-calling adapter trained in the companion notebook [FunctionGemma_270M_1_LoRA_Fine_Tuning.ipynb](https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_1_LoRA_Fine_Tuning.ipynb).

1. **Step 1 — Install dependencies:** the LiteRT-LM runtime (`litert-lm`), the PyTorch-to-LiteRT conversion stack (`litert-torch`, `ai-edge-quantizer`, `flatbuffers`), and Hugging Face libraries.
2. **Step 2 — Get the fine-tuned adapter:** use the files from a fine-tuning run in this session, or download them from the Hugging Face repository and subfolder you uploaded to (`HF_REPO_ID`, `HF_SUBFOLDER`).
3. **Step 3 — Export the base model & convert the adapter:**
   - Export `google/functiongemma-270m-it` to `.litertlm` (`llm_model_type: function_gemma`) with frozen INT8 base weights and open input slots for rank-32 LoRA tensors in the `decode` and all `prefill_{seq_len}` signatures.
   - Serialize the PEFT LoRA weights into the 64KB-aligned, memory-mapped `.tflite` FlatBuffer format expected by LiteRT-LM's C++ `LoraData` runtime.
4. **Step 4 — Verify LoRA signatures & input slots:** unpack the `.litertlm` container and confirm the dynamic LoRA slots are present.
5. **Step 5 — Side-by-side dynamic inference:** load the base `.litertlm` once and send the smart-home tools plus the two example requests with:
   - **Base (zero-LoRA):** no task adapter ($W_{base} + 0 = W_{base}$).
   - **+ task LoRA (dynamic):** `functiongemma_270m_task_lora.tflite` attached via `litert_lm.LoraConfig` ($W_{base} + B \cdot A = W_{task}$).
   - **Fused (ground truth):** the statically merged model from the companion notebook.

| Request | Expected tool call |
|---|---|
| *Turn on the living room lights* | `set_device_state(device_id="light.living_room", state="on")` |
| *I want the office at 22 degrees* | `set_temperature(zone_id="climate.office", celsius=22)` |

**Setup**:
- A standard **CPU** runtime is enough; no GPU is needed.
- **Estimated Runtime**: Expect **~20 minutes** end-to-end; most of it is the base-model conversion in Step 3.
- Add `HF_TOKEN` in Colab **Secrets (🔑)** to download the gated `google/functiongemma-270m-it` repository.

## Step 1: Install Dependencies

* **`litert-lm`**: The LiteRT-LM runtime and Python bindings (`litert_lm.Engine`, `litert_lm.LoraConfig`) used in Step 5 to load `.litertlm` models and attach `.tflite` LoRA adapters at runtime.
* **`litert-torch` & `ai-edge-quantizer`**: Used in Step 3 to trace `google/functiongemma-270m-it` with rank-32 LoRA input signatures and quantize the base weights to INT8.
* **`flatbuffers` & `safetensors`**: Used in Step 3 to read the trained PEFT `adapter_model.safetensors` weights and pack them into a 64KB-aligned, memory-mapped `.tflite` FlatBuffer.
* **`transformers` & `huggingface_hub`**: Used to download the base checkpoint and the fine-tuned adapter.

In [ ]:
# @title { display-mode: "form" }
!pip install -q --no-warn-conflicts \
    "litert-lm>=0.1.3" \
    "litert-torch>=0.1.0" \
    "ai-edge-torch>=0.4.0" \
    "torch>=2.4.0" \
    "torchvision" \
    "torchaudio" \
    "transformers>=4.44.0" \
    "huggingface_hub" \
    "safetensors" \
    "flatbuffers" \
    "protobuf>=4.25.3,<6.0.0"
!pip install -q --no-deps "typing-extensions>=4.13.0"

import importlib
import typing_extensions
importlib.reload(typing_extensions)
if not hasattr(typing_extensions, "Sentinel"):
    typing_extensions.Sentinel = object()

print("✅ Packages installed successfully.")

## Step 2: Get the Fine-Tuned Adapter

Uses the adapter, fused model, and `tools.json` from a run of `FunctionGemma_270M_1_LoRA_Fine_Tuning.ipynb` in this same session if they are present. Otherwise it downloads them from `HF_REPO_ID` / `HF_SUBFOLDER`: set these to the repository and individual subfolder you uploaded to in Step 11 of the fine-tuning notebook.

In [ ]:
# @title { display-mode: "form" }
import json
import os
import shutil
from huggingface_hub import hf_hub_download

# Files from a fine-tuning run in this same Colab session are used directly if present.
# Otherwise they are downloaded from HF_REPO_ID/HF_SUBFOLDER: the reference artifacts by default,
# or point these at the repo/subfolder you uploaded to in Step 11 of FunctionGemma_270M_1_LoRA_Fine_Tuning.ipynb.
HF_REPO_ID = "litert-community/functiongemma-270m-lora-litertlm"  # @param {type:"string"}
HF_SUBFOLDER = "functiongemma_270m_lora_gpu"  # @param {type:"string"}

WORK_DIR = "/content/functiongemma_lora_artifacts"
os.makedirs(WORK_DIR, exist_ok=True)

LOCAL_CANDIDATES = {
    "functiongemma_270m_lora/adapter_model.safetensors": "/content/functiongemma_270m_lora/adapter_model.safetensors",
    "functiongemma_270m_fused_litertlm/model.litertlm": "/content/functiongemma_270m_fused_litertlm/model.litertlm",
    "tools.json": "/content/sft_data/tools.json",
}


def fetch(filename: str) -> str:
    dst = os.path.join(WORK_DIR, filename)
    if os.path.isfile(dst):
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    local = LOCAL_CANDIDATES.get(filename)
    if local and os.path.isfile(local):
        shutil.copy2(local, dst)
        return dst
    subfolder = HF_SUBFOLDER.strip("/")
    path_in_repo = f"{subfolder}/{filename}" if subfolder else filename
    print(f"Downloading {path_in_repo} from {HF_REPO_ID} ...")
    shutil.copy2(hf_hub_download(repo_id=HF_REPO_ID, filename=path_in_repo), dst)
    return dst


ADAPTER_SAFETENSORS_PATH = fetch("functiongemma_270m_lora/adapter_model.safetensors")
FUSED_LITERTLM_PATH = fetch("functiongemma_270m_fused_litertlm/model.litertlm")
with open(fetch("tools.json"), encoding="utf-8") as f:
    TOOLS = json.load(f)
LORA_TFLITE_PATH = "/content/functiongemma_270m_task_lora.tflite"  # written in Step 3
ZERO_LORA_PATH = "/content/functiongemma_270m_zero_lora.tflite"  # written in Step 3

for label, path in [("LoRA adapter (.safetensors)", ADAPTER_SAFETENSORS_PATH), ("Fused model (.litertlm)", FUSED_LITERTLM_PATH)]:
    print(f"  - {label}: {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
print("  - Tools:", ", ".join(t["name"] for t in TOOLS))

## Step 3: Export the Base Model with Dynamic LoRA Input Signatures

1. **LiteRT-LM LoRA contract:** LiteRT-LM's C++ runtime (`llm_litert_compiled_model_executor.cc`) binds dynamic LoRA buffers to the canonical signatures `decode` and `prefill_{seq_len}`, and allocates them from the signature named `"decode"`.
2. **Canonical signature patch:** By default, `converter.py` creates signatures with `_lora_r{rank}` suffixes, which leaves `decode` without LoRA slots. A patch on `converter._add_signatures` gives the primary `prefill_*` and `decode` signatures the rank-32 LoRA input slots.
3. **FunctionGemma metadata:** The container is written with `llm_model_type: function_gemma`, the `<start_function_response>` stop token, and the FunctionGemma chat template, so `create_conversation(tools=...)` formats tools exactly as in training.
4. **Zero static baking:** Base model weights stay frozen; all 144 LoRA parameter tensors (`lora_atten_q_a_prime_weight_0`, etc.) remain open input slots.
5. **Memory-mapped LoRA serializer:** LiteRT-LM's `LoraData` requires external buffer offsets (`buffer.offset` and `buffer.size`) with 64KB alignment; `export_lora_to_litert_lm` writes compliant `.tflite` adapter files (the task adapter and a zero adapter for the baseline).

In [ ]:
# @title { display-mode: "form" }
import os
import sys
import types
import torch

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    from huggingface_hub import notebook_login
    notebook_login()

# ------------------------------------------------------------------------------
# 🔧 Python 3.13 Compatibility: Mock broken unused metrics C-wrapper
# ------------------------------------------------------------------------------
if 'tensorflow.lite.python.metrics._pywrap_tensorflow_lite_metrics_wrapper' not in sys.modules:
    mock_mod = types.ModuleType('tensorflow.lite.python.metrics._pywrap_tensorflow_lite_metrics_wrapper')
    class DummyMetricsWrapper:
        def __init__(self, *args, **kwargs): pass
        def ExportMetrics(self): return None
    mock_mod.MetricsWrapper = DummyMetricsWrapper
    sys.modules['tensorflow.lite.python.metrics._pywrap_tensorflow_lite_metrics_wrapper'] = mock_mod

# Bypass Protobuf runtime version check across conflicting packages
try:
    import google.protobuf.runtime_version
    google.protobuf.runtime_version.ValidateProtobufRuntimeVersion = lambda *args, **kwargs: None
    google.protobuf.runtime_version._ReportVersionError = lambda *args, **kwargs: None
except Exception:
    pass
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

from huggingface_hub import snapshot_download
from litert_torch.generative.examples.gemma3 import gemma3
try:
    from litert_torch.generative.layers import kv_cache
except ImportError:
    from ai_edge_torch.generative.layers import kv_cache
from litert_torch.generative.utilities import converter
from litert_torch.generative.utilities.export_config import ExportConfig
from litert_torch.generative.layers import lora as lora_utils

import litert_lm_builder
from litert_lm_builder import litertlm_builder as _litertlm_builder_mod
from litert_lm_builder.runtime.proto import llm_metadata_pb2 as _llm_metadata_pb2

if not hasattr(_litertlm_builder_mod, "TfLiteModelType"):
    class _CompatTfLiteModelType:
        PREFILL_DECODE = _llm_metadata_pb2.LlmMetadata.TF_LITE_PREFILL_DECODE
        EMBEDDER = _llm_metadata_pb2.LlmMetadata.TF_LITE_EMBEDDER
        PER_LAYER_EMBEDDER = _llm_metadata_pb2.LlmMetadata.TF_LITE_PER_LAYER_EMBEDDER
        VISION_ENCODER = _llm_metadata_pb2.LlmMetadata.TF_LITE_VISION_ENCODER
        VISION_ADAPTER = _llm_metadata_pb2.LlmMetadata.TF_LITE_VISION_ADAPTER
        END_OF_VISION = _llm_metadata_pb2.LlmMetadata.TF_LITE_END_OF_VISION
        AUDIO_ENCODER_HW = _llm_metadata_pb2.LlmMetadata.TF_LITE_AUDIO_ENCODER_HW
        AUDIO_ADAPTER = _llm_metadata_pb2.LlmMetadata.TF_LITE_AUDIO_ADAPTER
        END_OF_AUDIO = _llm_metadata_pb2.LlmMetadata.TF_LITE_END_OF_AUDIO
        AUX = _llm_metadata_pb2.LlmMetadata.TF_LITE_AUX
    _litertlm_builder_mod.TfLiteModelType = _CompatTfLiteModelType
    litert_lm_builder.TfLiteModelType = _CompatTfLiteModelType

BASE_MODEL_ID = "google/functiongemma-270m-it"
OUTPUT_DIR = "/content/functiongemma_270m_dynamic_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📥 Downloading official weights for {BASE_MODEL_ID}...")
try:
    checkpoint_dir = snapshot_download(repo_id=BASE_MODEL_ID)
except Exception:
    print("ℹ️ Falling back to ungated mirror 'unsloth/functiongemma-270m-it'...")
    checkpoint_dir = snapshot_download(repo_id="unsloth/functiongemma-270m-it")
print(f"✅ Checkpoint downloaded to: {checkpoint_dir}")

print("🏗️ Building PyTorch 270M backbone architecture...")
pytorch_model = gemma3.build_model_270m(checkpoint_dir)

print("🔧 Patching Gemma3 Decoder to support dynamic LoRA forward pass...")
from litert_torch.generative.examples.gemma3 import decoder as gemma3_decoder
rotary_pos_emb = gemma3_decoder.rotary_pos_emb
kv_utils = gemma3_decoder.kv_utils

def patched_block_forward(self, x, rope=None, mask=None, input_pos=None, kv_cache=None, lora=None):
    x_norm = self.pre_atten_norm(x)
    attn_out, kv = self.atten_func(x_norm, rope, mask, input_pos, kv_cache, lora)
    attn_out_norm = self.post_atten_norm(attn_out)
    x = x + attn_out_norm
    output = x + self.ff(x)
    return output, kv

def patched_decoder_forward(self, tokens, input_pos, kv_cache, input_embeds=None, mask=None, image_indices=None, lora=None, export_config=None):
    if input_embeds is None:
        input_embeds = self.tok_embedding(tokens)
        if self.config.embedding_scale is not None:
            input_embeds = input_embeds * self.config.embedding_scale
    attn_config = self.config.block_config(0).attn_config
    rope = [
        rotary_pos_emb.build_rope(
            input_pos, attn_config.head_dim, self.config.block_config(i).attn_config.rotary_base
        ) for i in range(self.config.num_layers)
    ]
    if mask is None:
        assert self.mask_cache is not None, 'Mask cache must be built.'
        assert kv_cache is not None, 'KV cache must be provided.'
        mask = [self.mask_cache.index_select(2, input_pos) for _ in range(self.config.num_layers)]
        kv_cache_max_len = kv_cache.get_max_seq_len()
    else:
        kv_cache_max_len = mask.size(3)
    pixel_mask = self.build_pixel_mask(image_indices, kv_cache_max_len) if image_indices is not None else None
    return self._forward_with_embeds(input_embeds, rope, mask, input_pos, kv_cache, pixel_mask, lora, export_config)

def patched_fwd_embeds(self, input_embeds, rope, mask, input_pos, kv_cache, pixel_mask=None, lora=None, export_config=None):
    assert len(self.transformer_blocks) == len(kv_cache.caches)
    x = input_embeds
    if pixel_mask is None:
        mask = [
            self.get_local_global_attention_mask(
                mask[i] if isinstance(mask, list) else mask,
                self.config.block_config(i).attn_config.attn_type,
                input_pos,
                self.config.block_config(i).attn_config.sliding_window_size,
            ) for i in range(self.config.num_layers)
        ]
    else:
        pixel_mask = pixel_mask.index_select(2, input_pos)
        mask = [
            self.compose_mask(
                mask[i] if isinstance(mask, list) else mask,
                pixel_mask,
                self.config.block_config(i).attn_config.attn_type,
            ) for i in range(self.config.num_layers)
        ]
    updated_kv_entries = []
    for i, block in enumerate(self.transformer_blocks):
        mask_entry = mask[i] if isinstance(mask, list) else mask
        kv_entry = kv_cache.caches[i] if kv_cache else None
        lora_adapter = lora.adapters[i] if lora is not None else None
        x, kv_entry = block(x, rope[i], mask_entry, input_pos, kv_entry, lora=lora_adapter)
        if kv_entry:
            updated_kv_entries.append(kv_entry)
    updated_kv_cache = kv_utils.KVCache(tuple(updated_kv_entries))
    if export_config is not None and torch.numel(input_pos) > 1 and not export_config.output_logits_on_prefill:
        return {'kv_cache': updated_kv_cache}
    x = self.final_norm(x)
    return {'logits': self.lm_head(x), 'kv_cache': updated_kv_cache}

gemma3_decoder.DecoderBlock.forward = patched_block_forward
gemma3_decoder.Decoder.forward = torch.inference_mode(patched_decoder_forward)
gemma3_decoder.Decoder._forward_with_embeds = patched_fwd_embeds
print("✅ Gemma3 Decoder successfully patched with LoRA support.")

# Monkeypatch converter._add_signatures to export canonical signature names
# ('prefill_{seq_len}' and 'decode' WITHOUT '_lora_r32' suffix) equipped with LoRA input slots.
original_add_signatures = converter._add_signatures

def patched_add_signatures(
    converter_obj,
    pytorch_model,
    prefill_seq_lens,
    kv_cache_max_len,
    pixel_values_size,
    pixel_seq_len,
    config,
    loras,
    export_config,
    signature_prefix='',
):
    active_lora = None
    for l in loras:
        if l is not None:
            active_lora = l
            break

    if active_lora is not None:
        print(f"🔧 [Patch] Injecting canonical LoRA signatures: prefill_* and decode (rank={active_lora.get_rank()})...")
        prefill_tokens_list = [
            torch.full((1, seq_len), 0, dtype=torch.int) for seq_len in prefill_seq_lens
        ]
        prefill_input_pos_list = [
            torch.arange(0, seq_len, dtype=torch.int) for seq_len in prefill_seq_lens
        ]
        prefill_masks = None
        if export_config.mask_as_input:
            prefill_masks = converter._build_mask(
                prefill_seq_lens, kv_cache_max_len, config.causal_mask_value
            )
            if not isinstance(prefill_masks, list):
                prefill_masks = [prefill_masks]

        decode_token = torch.tensor([[0] for _ in range(export_config.decode_batch_size)], dtype=torch.int)
        decode_input_pos = torch.tensor([0], dtype=torch.int)
        prefill_kv = converter.kv_utils.KVCache.from_model_config(
            kv_cache_max_len, config, kv_layout=export_config.kvcache_layout
        )
        decode_kv = converter.kv_utils.KVCache.from_model_config(
            kv_cache_max_len, config, batch_size=export_config.decode_batch_size, kv_layout=export_config.kvcache_layout
        )
        mod = converter.ExportableModule(pytorch_model, export_config=export_config).eval()

        # Add canonical prefill signatures with LoRA inputs
        for i, seq_len in enumerate(prefill_seq_lens):
            sample_kwargs = {
                'tokens': prefill_tokens_list[i],
                'input_pos': prefill_input_pos_list[i],
                'kv_cache': prefill_kv,
                'lora': active_lora,
            }
            if prefill_masks is not None:
                sample_kwargs['mask'] = prefill_masks[i]
            converter_obj.add_signature(
                f'{signature_prefix}prefill_{seq_len}',
                mod,
                sample_kwargs=sample_kwargs,
            )

        # Add canonical decode signature with LoRA inputs
        decode_kwargs = {
            'tokens': decode_token,
            'input_pos': decode_input_pos,
            'kv_cache': decode_kv,
            'lora': active_lora,
        }
        if export_config.mask_as_input:
            decode_kwargs['mask'] = converter._build_mask(
                1, kv_cache_max_len, config.causal_mask_value
            )
        converter_obj.add_signature(
            f'{signature_prefix}decode',
            mod,
            sample_kwargs=decode_kwargs,
        )
    else:
        original_add_signatures(
            converter_obj,
            pytorch_model,
            prefill_seq_lens,
            kv_cache_max_len,
            pixel_values_size,
            pixel_seq_len,
            config,
            loras,
            export_config,
            signature_prefix,
        )

converter._add_signatures = patched_add_signatures
print("✅ Canonical LoRA signature exporter installed.")

# Setup export configuration
export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

# Mark the container as FunctionGemma (`llm_model_type: function_gemma`) so LiteRT-LM formats the tools
# with FunctionGemmaDataProcessor, stops at `<start_function_response>`, and applies the FunctionGemma chat
# template. The template is patched so its format_function_declaration macro accepts the pre-formatted
# "declaration:..." strings LiteRT-LM passes in.
import json
import re
import jinja2
from litert_torch.generative.utilities import litertlm_builder as _lt_litertlm_builder

FC_STR_START = "{%- if tool_data is string -%}{{- tool_data -}}{%- else -%}"
FC_STR_END = "{%- endif -%}{#- fc-string-tool -#}"


def _patch_template_text(tmpl: str) -> str:
    hdr = "{% macro format_function_declaration(tool_data) -%}"
    hdr_idx = tmpl.find(hdr)
    if hdr_idx == -1 or FC_STR_END in tmpl:
        return tmpl
    body_start = hdr_idx + len(hdr)
    end_match = re.compile(r"\{%-?\s*endmacro").search(tmpl, body_start)
    if not end_match:
        return tmpl
    body = tmpl[body_start:end_match.start()]
    if FC_STR_START not in body:
        body = body.replace("declaration:", FC_STR_START + "declaration:", 1)
    patched = tmpl[:body_start] + body + FC_STR_END + tmpl[end_match.start():]
    jinja2.Environment().parse(patched)
    return patched


def _load_chat_template(model_dir: str) -> str | None:
    jinja_path = os.path.join(model_dir, "chat_template.jinja")
    if os.path.isfile(jinja_path):
        with open(jinja_path, encoding="utf-8") as f:
            return f.read()
    with open(os.path.join(model_dir, "tokenizer_config.json"), encoding="utf-8") as f:
        tmpl = json.load(f).get("chat_template")
    return tmpl if isinstance(tmpl, str) else None


FUNCTION_GEMMA_TEMPLATE = _load_chat_template(checkpoint_dir)
if FUNCTION_GEMMA_TEMPLATE:
    FUNCTION_GEMMA_TEMPLATE = _patch_template_text(FUNCTION_GEMMA_TEMPLATE)

_FileBuilder = _lt_litertlm_builder.litertlm_builder.LitertLmFileBuilder
# Keep the unpatched method so re-running this cell (or unpacking in Step 4) never stacks patches.
_orig_add_llm_metadata = getattr(_FileBuilder, "_fg_orig_add_llm_metadata", _FileBuilder.add_llm_metadata)
_FileBuilder._fg_orig_add_llm_metadata = _orig_add_llm_metadata
_FileBuilder.add_llm_metadata = _orig_add_llm_metadata


def _fg_add_llm_metadata(self, llm_metadata_path, *args, **kwargs):
    """Rewrites the binary LlmMetadata written by the converter before it is packed."""
    llm_metadata = _lt_litertlm_builder.llm_metadata_pb2.LlmMetadata()
    with open(llm_metadata_path, "rb") as f:
        llm_metadata.ParseFromString(f.read())
    llm_metadata.llm_model_type.function_gemma.SetInParent()
    if "<start_function_response>" not in {st.token_str for st in llm_metadata.stop_tokens if st.HasField("token_str")}:
        llm_metadata.stop_tokens.add().token_str = "<start_function_response>"
    if FUNCTION_GEMMA_TEMPLATE:
        llm_metadata.jinja_prompt_template = FUNCTION_GEMMA_TEMPLATE
    with open(llm_metadata_path, "wb") as f:
        f.write(llm_metadata.SerializeToString())
    return _orig_add_llm_metadata(self, llm_metadata_path, *args, **kwargs)

# Metadata and tokenizer configuration for LiteRT-LM
litertlm_config = {
    "tokenizer_model_path": os.path.join(checkpoint_dir, "tokenizer.model"),
    "start_token_id": 2,  # <bos>
    "stop_token_ids": [1, 106],  # [<eos>, <end_of_turn>]
    "user_prompt_prefix": "<start_of_turn>user\n",
    "user_prompt_suffix": "<end_of_turn>\n",
    "model_prompt_prefix": "<start_of_turn>model\n",
    "model_prompt_suffix": "<end_of_turn>\n",
    "output_format": "litertlm",
}

existing_litertlm = [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR) if f.endswith('.litertlm')] if os.path.exists(OUTPUT_DIR) else []
if existing_litertlm:
    BASE_LITERTLM_PATH = existing_litertlm[0]
    print(f"⚡ Found existing base model with LoRA slots: {BASE_LITERTLM_PATH} ({os.path.getsize(BASE_LITERTLM_PATH)/1e6:.2f} MB)")
    print("   Skipping 6-minute conversion step!")
else:
    print("⚡ Converting to LiteRT-LM with dynamic LoRA signatures (rank=32)...")
    # Only patch the LlmMetadata while this conversion packs the container.
    _FileBuilder.add_llm_metadata = _fg_add_llm_metadata
    BASE_LITERTLM_PATH = os.path.join(OUTPUT_DIR, "functiongemma_270m_dynamic_lora.litertlm")
    converter.convert_to_litert(
        pytorch_model,
        output_path=OUTPUT_DIR,
        output_name_prefix="functiongemma_270m_dynamic_lora",
        prefill_seq_len=[128, 256, 512, 1024, 2048],
        kv_cache_max_len=2048,
        quantize="dynamic_int8",
        lora_ranks=[32],  # <-- INJECTS DYNAMIC LORA INPUT SIGNATURES
        export_config=export_config,
        **litertlm_config
    )
    _FileBuilder.add_llm_metadata = _orig_add_llm_metadata
    for fname in os.listdir(OUTPUT_DIR):
        if fname.endswith(".litertlm"):
            BASE_LITERTLM_PATH = os.path.join(OUTPUT_DIR, fname)
            break
    print(f"✅ Base model with Dynamic LoRA slots exported: {BASE_LITERTLM_PATH} ({os.path.getsize(BASE_LITERTLM_PATH)/1e6:.2f} MB)")

# ==============================================================================
# 🎯 LiteRT-LM Memory-Mapped LoRA FlatBuffer Serializer
# ==============================================================================
# LiteRT-LM's LoraData (runtime/util/lora_data.cc)
# loads tensors via zero-copy mmap using: ReadData(buffer->offset(), buffer->size()).
# Raw tensor bytes are appended after the FlatBuffer header at 64KB alignment.
# ==============================================================================
def export_lora_to_litert_lm(lora_obj, output_path, base_offset=65536):
    """Serializes LoRA weights into the memory-mapped FlatBuffer format required by LiteRT-LM."""
    builder = lora_utils.flatbuffers.Builder(4096)
    schema_fb = lora_utils.schema_fb

    tensors, (names, _) = lora_utils._flatten_lora(lora_obj)
    names = [f"lora_{name}" for name in names]
    rank = lora_obj.get_rank()

    buffer_offsets = []
    # Buffer 0: empty buffer for convention
    schema_fb.BufferStart(builder)
    buffer_offsets.append(schema_fb.BufferEnd(builder))

    cur_offset = base_offset
    raw_bytes_list = []
    for tensor in tensors:
        t_bytes = tensor.detach().cpu().to(torch.float32).numpy().tobytes()
        raw_bytes_list.append(t_bytes)
        schema_fb.BufferStart(builder)
        schema_fb.BufferAddOffset(builder, cur_offset)
        schema_fb.BufferAddSize(builder, len(t_bytes))
        buffer_offsets.append(schema_fb.BufferEnd(builder))
        cur_offset += len(t_bytes)

    schema_fb.ModelStartBuffersVector(builder, len(buffer_offsets))
    for b_off in reversed(buffer_offsets):
        builder.PrependUOffsetTRelative(b_off)
    buffers_offset = builder.EndVector()

    tensor_offsets = []
    for i, (name, tensor) in enumerate(zip(names, tensors)):
        tensor_offsets.append(lora_utils._add_tensor(builder, name, tensor.shape, i + 1))

    schema_fb.SubGraphStartTensorsVector(builder, len(tensor_offsets))
    for t_off in reversed(tensor_offsets):
        builder.PrependUOffsetTRelative(t_off)
    tensors_offset = builder.EndVector()

    string_offset = builder.CreateString("lora_params")
    schema_fb.SubGraphStart(builder)
    schema_fb.SubGraphAddName(builder, string_offset)
    schema_fb.SubGraphAddTensors(builder, tensors_offset)
    subgraph_offset = schema_fb.SubGraphEnd(builder)

    schema_fb.ModelStartSubgraphsVector(builder, 1)
    builder.PrependUOffsetTRelative(subgraph_offset)
    subgraphs_offset = builder.EndVector()

    metadata_name_offset = builder.CreateString("lora_rank")
    schema_fb.MetadataStart(builder)
    schema_fb.MetadataAddName(builder, metadata_name_offset)
    schema_fb.MetadataAddBuffer(builder, rank)
    metadata_offset = schema_fb.MetadataEnd(builder)

    schema_fb.ModelStartMetadataVector(builder, 1)
    builder.PrependUOffsetTRelative(metadata_offset)
    metadata_vector_offset = builder.EndVector()

    model_desc = builder.CreateString("lora_params")
    schema_fb.ModelStart(builder)
    schema_fb.ModelAddVersion(builder, 3)
    schema_fb.ModelAddDescription(builder, model_desc)
    schema_fb.ModelAddBuffers(builder, buffers_offset)
    schema_fb.ModelAddSubgraphs(builder, subgraphs_offset)
    schema_fb.ModelAddMetadata(builder, metadata_vector_offset)
    model_offset = schema_fb.ModelEnd(builder)
    builder.Finish(model_offset, file_identifier=b"TFL3")
    flatbuffer_header = bytearray(builder.Output())

    if len(flatbuffer_header) > base_offset:
        raise ValueError(f"FlatBuffer header ({len(flatbuffer_header)} bytes) exceeds base_offset ({base_offset} bytes)")

    with open(output_path, "wb") as f:
        f.write(flatbuffer_header)
        f.write(b"\x00" * (base_offset - len(flatbuffer_header)))
        for raw_bytes in raw_bytes_list:
            f.write(raw_bytes)
    print(f"✅ Generated LiteRT-LM LoRA FlatBuffer: {output_path} ({os.path.getsize(output_path)/1e6:.2f} MB, rank={rank})")

def convert_inmemory_lora_to_litert_lm(input_path, output_path):
    """Repacks an in-memory/standard TFLite LoRA into a 64KB page-aligned LiteRT-LM FlatBuffer."""
    try:
        with open(input_path, "rb") as f:
            buf = f.read()
        if len(buf) <= 8 or buf[4:8] != b"TFL3":
            return False
        model = lora_utils.schema_fb.Model.GetRootAsModel(buf, 0)
        subgraph = model.Subgraphs(0)
        tensors_dict = {}
        for i in range(subgraph.TensorsLength()):
            t = subgraph.Tensors(i)
            t_name = t.Name().decode('utf-8')
            b_idx = t.Buffer()
            buffer_obj = model.Buffers(b_idx)
            raw_data = buffer_obj.DataAsNumpy()
            if raw_data is not None and len(raw_data) > 0:
                tensors_dict[t_name] = torch.from_numpy(raw_data.view(np.float32)).reshape(tuple(t.ShapeAsNumpy()))
        if not tensors_dict:
            return False
        # Rebuild LoRA object and re-export
        repacked_lora = lora_utils.LoRA._from_tensor_generator(
            tensor_generator=lambda shape, dtype: torch.zeros(shape, dtype=dtype),
            rank=32,
            config=pytorch_model.config,
        )
        tensors, (names, _) = lora_utils._flatten_lora(repacked_lora)
        for name, tensor in zip(names, tensors):
            key = f"lora_{name}"
            if key in tensors_dict and tensors_dict[key].shape == tensor.shape:
                tensor.copy_(tensors_dict[key])
        export_lora_to_litert_lm(repacked_lora, output_path)
        print(f"🎉 Successfully repacked {input_path} into LiteRT-LM 64KB-aligned format at {output_path}!")
        return True
    except Exception as e:
        print(f"⚠️ Could not repack existing .tflite: {e}")
        return False

# ------------------------------------------------------------------------------
# 1. Export Baseline Zero-LoRA (for unadapted base model inference)
# ------------------------------------------------------------------------------
print("\n⚙️ Exporting baseline Zero-LoRA adapter (delta = 0)...")
zero_lora = lora_utils.LoRA.zeros(rank=32, config=pytorch_model.config)
export_lora_to_litert_lm(zero_lora, ZERO_LORA_PATH)

# ------------------------------------------------------------------------------
# 2. Export / Convert Task LoRA Adapter
# ------------------------------------------------------------------------------
converted = False

# Pathway A: Direct conversion from real adapter_model.safetensors
if 'ADAPTER_SAFETENSORS_PATH' in globals() and ADAPTER_SAFETENSORS_PATH and os.path.exists(ADAPTER_SAFETENSORS_PATH):
    try:
        import safetensors
        print(f"\n🎯 Found real fine-tuned adapter weights: {ADAPTER_SAFETENSORS_PATH}")
        print("   Converting directly to LiteRT-LM memory-mapped FlatBuffer...")
        with safetensors.safe_open(ADAPTER_SAFETENSORS_PATH, framework="pt") as f:
            keys = list(f.keys())
            sample_key = next((k for k in keys if "q_proj.lora_A.weight" in k), None)

        if sample_key and ".self_attn." in sample_key:
            prefix = sample_key.split(".self_attn.")[0].rsplit(".", 1)[0] + ".{}"
        else:
            prefix = "base_model.model.model.layers.{}"

        print(f"   Detected LoRA layer pattern: {prefix}")
        tensor_names = lora_utils.LoRATensorNames(
            attn_query_w_a = f"{prefix}.self_attn.q_proj.lora_A.weight",
            attn_query_w_b = f"{prefix}.self_attn.q_proj.lora_B.weight",
            attn_key_w_a   = f"{prefix}.self_attn.k_proj.lora_A.weight",
            attn_key_w_b   = f"{prefix}.self_attn.k_proj.lora_B.weight",
            attn_value_w_a = f"{prefix}.self_attn.v_proj.lora_A.weight",
            attn_value_w_b = f"{prefix}.self_attn.v_proj.lora_B.weight",
            attn_output_w_a = f"{prefix}.self_attn.o_proj.lora_A.weight",
            attn_output_w_b = f"{prefix}.self_attn.o_proj.lora_B.weight",
        )

        trained_lora = lora_utils.LoRA.from_safetensors(
            path = ADAPTER_SAFETENSORS_PATH,
            scale = 2.0,  # alpha=64 / rank=32
            config = pytorch_model.config,
            lora_tensor_names = tensor_names,
        )
        export_lora_to_litert_lm(trained_lora, LORA_TFLITE_PATH)
        converted = True
        print(f"🎉 Real fine-tuned LoRA successfully converted to {LORA_TFLITE_PATH}!")
    except Exception as e:
        print(f"⚠️ Failed to convert from safetensors ({e}). Falling back...")
        converted = False

# Pathway B: Verify if existing .tflite is already 64KB memory-mapped
if not converted and os.path.exists(LORA_TFLITE_PATH) and os.path.getsize(LORA_TFLITE_PATH) > 0:
    try:
        with open(LORA_TFLITE_PATH, "rb") as f:
            full_file_buf = f.read()
        if len(full_file_buf) > 8 and full_file_buf[4:8] == b"TFL3":
            model_fb = lora_utils.schema_fb.Model.GetRootAsModel(full_file_buf, 0)
            if model_fb.BuffersLength() > 1 and model_fb.Buffers(1).Offset() >= 65536 and model_fb.Buffers(1).Size() > 0:
                last_b = model_fb.Buffers(model_fb.BuffersLength() - 1)
                if last_b.Offset() + last_b.Size() <= len(full_file_buf):
                    converted = True
                    print(f"✅ Existing task LoRA {LORA_TFLITE_PATH} verified in LiteRT-LM memory-mapped format.")
    except Exception as e:
        print(f"⚠️ Inspection of {LORA_TFLITE_PATH} encountered: {e}")
        converted = False

# Pathway C: Fallback to synthetic LoRA for pipeline testing if no real adapter provided
if not converted:
    print(f"\n⚙️ Generating dynamic task LoRA adapter with compliant memory-mapped format...")
    print("   (Notice: No valid memory-mapped adapter was found. A synthetic rank-32 adapter is created for pipeline verification.)")
    print("   👉 To use real fine-tuned weights, upload 'adapter_model.safetensors' to /content/ and re-run Step 3.")
    task_lora = lora_utils.LoRA._from_tensor_generator(
        tensor_generator=lambda shape, dtype: torch.randn(shape, dtype=dtype) * 0.05,
        rank=32,
        config=pytorch_model.config,
    )
    export_lora_to_litert_lm(task_lora, LORA_TFLITE_PATH)

## Step 4: Verify LoRA Signatures & Input Slots

Unpacks the generated `.litertlm` file and checks that the canonical signatures (`decode` and `prefill_*`) have the open LoRA input slots (183 inputs each: 39 standard inputs + 144 LoRA parameter tensors).

In [ ]:
# @title { display-mode: "form" }
import os
import sys
import types
import glob
import tempfile

if 'tensorflow.lite.python.metrics._pywrap_tensorflow_lite_metrics_wrapper' not in sys.modules:
    mock_mod = types.ModuleType('tensorflow.lite.python.metrics._pywrap_tensorflow_lite_metrics_wrapper')
    class DummyMetricsWrapper:
        def __init__(self, *args, **kwargs): pass
        def ExportMetrics(self): return None
    mock_mod.MetricsWrapper = DummyMetricsWrapper
    sys.modules['tensorflow.lite.python.metrics._pywrap_tensorflow_lite_metrics_wrapper'] = mock_mod

import tensorflow as tf
from litert_lm_builder import LitertLmFileBuilder

print(f"🔍 Inspecting signatures inside: {BASE_LITERTLM_PATH}")
with tempfile.TemporaryDirectory() as tmpdir:
    LitertLmFileBuilder.unpack(BASE_LITERTLM_PATH, tmpdir)
    tflite_files = glob.glob(os.path.join(tmpdir, "*.tflite"))
    if not tflite_files:
        raise RuntimeError("No .tflite model found in unpacked container")
    tflite_extracted = tflite_files[0]
    interpreter = tf.lite.Interpreter(model_path=tflite_extracted)
    signatures = interpreter.get_signature_list()

print("\n📋 Available Model Signatures:")
lora_equipped_signatures = []
for sig_name, sig_info in signatures.items():
    inputs = sig_info.get("inputs", [])
    lora_inputs = [inp for inp in inputs if "lora" in inp.lower()]
    print(f"  • Signature: {sig_name:<20} Total Inputs: {len(inputs):<4} (LoRA Slots: {len(lora_inputs)})")
    if lora_inputs:
        lora_equipped_signatures.append(sig_name)

if lora_equipped_signatures:
    print(f"\n🎯 SUCCESS: {len(lora_equipped_signatures)} signatures contain active LoRA input slots!")
    print(f"   Canonical 'decode' signature has {len(signatures['decode']['inputs'])} inputs ready for LiteRT-LM LoraManager.")
else:
    print("⚠️ Warning: No LoRA input slots detected in model signatures.")

## Step 5: Side-by-Side Dynamic Inference in LiteRT-LM

Loads the base model once and sends the smart-home tools with each example request:

1. **Base (zero-LoRA):** a zero adapter fills the LoRA input slots, giving the unadapted base model ($W_{base} + 0 = W_{base}$).
2. **+ task LoRA (dynamic):** `LoraConfig(lora_path=LORA_TFLITE_PATH)` attaches the fine-tuned rank-32 weights ($W_{base} + B \cdot A = W_{task}$).
3. **Fused (ground truth):** the statically merged model from the fine-tuning notebook, for comparison.

Each call is marked ✅ when the function and arguments match the expected call.

In [ ]:
import json
import os
import litert_lm

BASE_DYNAMIC_PATH = globals().get("BASE_LITERTLM_PATH", "/content/functiongemma_270m_dynamic_lora/functiongemma_270m_dynamic_lora.litertlm")
# The same held-out prompts as the other FunctionGemma Colabs.
DEMO_EXAMPLES = [
    {"query": "Turn on the living room lights", "name": "set_device_state", "arguments": {"device_id": "light.living_room", "state": "on"}},
    {"query": "I want the office at 22 degrees", "name": "set_temperature", "arguments": {"zone_id": "climate.office", "celsius": 22}},
]


class SmartHomeTool(litert_lm.Tool):
    """Passes the exact OpenAPI schema used for training to LiteRT-LM."""

    def __init__(self, definition):
        self._definition = definition

    def get_tool_description(self):
        return {"type": "function", "function": self._definition}

    def execute(self, param):
        return "ok"


def first_tool_call(reply):
    calls = reply.get("tool_calls") if isinstance(reply, dict) else getattr(reply, "tool_calls", None)
    if not calls:
        return None, None
    call = calls[0]
    if not isinstance(call, dict):  # litert_lm.ToolCall-style object
        return getattr(call, "name", None), dict(getattr(call, "arguments", {}) or {})
    fn = call.get("function", call)
    args = fn.get("arguments", {})
    if isinstance(args, str):
        args = json.loads(args or "{}")
    return fn.get("name"), dict(args)


def norm_args(args):
    return {k: str(v).strip().lower() for k, v in (args or {}).items()}


def run_demo(engine, lora_path=None):
    """Sends each demo prompt with the smart-home tools, optionally attaching a LoRA adapter."""
    kwargs = {"tools": tools, "automatic_tool_calling": False, "max_output_tokens": 64}
    if lora_path:
        kwargs["lora_config"] = litert_lm.LoraConfig(lora_path=lora_path)
    calls = []
    for row in DEMO_EXAMPLES:
        with engine.create_conversation(**kwargs) as conv:
            name, args = first_tool_call(conv.send_message(row["query"]))
        calls.append((name, args, name == row["name"] and norm_args(args) == norm_args(row["arguments"])))
    return calls


cpu_backend = litert_lm.Backend.CPU() if callable(getattr(litert_lm.Backend, "CPU", None)) else litert_lm.Backend.CPU
tools = [SmartHomeTool(t) for t in TOOLS]
results = {}

print(f"Loading the dynamic-LoRA base model once: {BASE_DYNAMIC_PATH}")
with litert_lm.Engine(
    model_path=BASE_DYNAMIC_PATH,
    backend=cpu_backend,
    max_num_tokens=1024,
    lora_rank_config=litert_lm.LoraRankConfig(lora_rank=32),
    cache_dir=":nocache",
) as engine:
    results["Base (zero-LoRA)"] = run_demo(engine, ZERO_LORA_PATH)
    results["+ task LoRA (dynamic)"] = run_demo(engine, LORA_TFLITE_PATH)

if os.path.isfile(FUSED_LITERTLM_PATH):
    with litert_lm.Engine(model_path=FUSED_LITERTLM_PATH, backend=cpu_backend, cache_dir=":nocache") as engine:
        results["Fused (ground truth)"] = run_demo(engine)

print("#" * 80 + "\n📊 3-WAY SIDE-BY-SIDE (SxS) INFERENCE SUMMARY\n" + "#" * 80)
for i, row in enumerate(DEMO_EXAMPLES):
    print("=" * 80 + f"\nPROMPT:   {row['query']}\nEXPECTED: {row['name']}({json.dumps(row['arguments'])})")
    for label, calls in results.items():
        name, args, correct = calls[i]
        shown = f"{name}({json.dumps(args)})" if name else "(no tool call)"
        print(f"  {'✅' if correct else '❌'} {label:<22} {shown}")

## Next Steps: Run Your `.litertlm` Model Everywhere

Once you have exported your `.litertlm` model, you can deploy and test it across mobile, web, and desktop:

- **[LiteRT-LM CLI (`litert-lm`)](https://developers.google.com/edge/litert-lm/cli)**: Run, benchmark, and serve `.litertlm` models from your terminal (`pip install litert-lm && litert-lm run model.litertlm`).
- **[OpenAI-Compatible Server](https://developers.google.com/edge/litert-lm/cli/openai_server#sending_chat_completion_requests)**: Serve the model with `litert-lm serve` and send it chat-completion requests, including tools.
- **[LiteRT-LM Web Demo](https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html)**: Upload your `.litertlm` package in Chrome or Edge to run 100% client-side in the browser with WebGPU acceleration.
- **[Google AI Edge Gallery App](https://developers.google.com/edge/gallery)**: Import your `.litertlm` model on Android or iOS (via your Hugging Face download URL or QR code) for interactive on-device inference.